# Gerador de Jogos de Sudoku
### Segunda Prova de Algebra Linear Computacional — UFRRJ, Prof. Marcio Sa


Este notebook implementa um gerador de jogos de Sudoku 9x9, junto com suas
respostas, conforme pedido na Questao 2 do trabalho. 

O código foi desenvolvido inicialmente como um script Python no VSCode (no repositório do projeto, scripts/sudoku-generator.py), para trabalhar no ambiente de desenvolvimento padrão e ter espaço para testar e estudar o que foi necessário ao longo do projeto, por exemplo, a heurística MRV (minimum remaining values) usada no solver extra, que eu não conhecia antes de pesquisar para esta questão. Só depois desse script estar completo, testado e eu ter entendido o conteúdo, fiz a reorganização neste notebook, dividindo em células de explicação e código, para atender ao formato .ipynb solicitado pelo professor.

## Hipoteses assumidas

- **1.** O Sudoku gerado e o classico 9x9, dividido em 9 blocos 3x3, com a
  regra padrao: cada linha, cada coluna e cada bloco 3x3 deve conter os
  numeros 1-9 sem repeticao.
- **2.** A **geracao** do tabuleiro completo (a "resposta") **nao** usa
  backtracking. Em vez disso, usa uma formula matematica fechada para
  construir um tabuleiro base valido e depois aplica uma sequencia de
  transformacoes que preservam validade (permutar simbolos, trocar linhas
  dentro de uma faixa de 3, trocar colunas dentro de uma banda de 3, trocar
  faixas de linhas entre si, trocar bandas de colunas entre si, e por fim
  transpor o tabuleiro). Essa abordagem e O(1) — nao ha tentativa e erro —
  e e a tecnica classica usada para gerar Sudokus aleatorios instantaneamente.
- **3.** A **remocao** de celulas (para criar o "jogo" que o usuario
  resolve) e feita removendo aleatoriamente uma quantidade definida pela
  dificuldade escolhida:
    - `facil` → 35 celulas removidas (~43% do tabuleiro vazio)
    - `medio` → 45 celulas removidas (~56%)
    - `dificil` → 55 celulas removidas (~68%)
  
  Hipotese simplificadora: não garantimos que o jogo resultante tenha
  solucao **única** apos a remocao. Apenas garantimos que a remoção não
  altera a validade do tabuleiro completo (a resposta). Como a resposta e
  entregue junto do jogo, esse ponto é trivial na pratica para os fins
  da prova.
- **4.** O ponto de entrada padrao do notebook e a funcao `testes()`
  seguida de `demonstracao()`, que rodam sem pedir nenhum input do usuario.
  A funcao `main()` oferece o modo interativo (pede quantidade de jogos e
  dificuldade via `input()`).

## Sobre o uso de LLM

Este codigo foi desenvolvido com auxilio do Claude (Anthropic) para:
- Revisar a corretude das transformacoes (cada uma foi verificada nos testes
  automatizados abaixo, conferindo que o tabuleiro permanece valido apos
  cada operacao)
- Organizar a impressao do tabuleiro e o fluxo interativo/de teste.
- Sugestão de adicionar o extra de Solver via backtracking. Não tinha pensado nessa possibilidade antes mas, após a sugestão do Claude de adicionar essa feature, vi que seria um aprendizado a mais bem interessante, já que nunca tinha implementado a heurística MRV. 

O raciocinio (por que cada transformacao preserva a validade de um Sudoku)
foi compreendido e validado manualmente antes da entrega.

## Como funciona a geracao (resumo da ideia matematica)

Um tabuleiro-base sempre valido pode ser construido com a formula:

```
base[i][j] = ( deslocamento(i) + j ) mod 9 + 1
deslocamento(i) = 3 * (i mod 3) + (i // 3)
```

Essa formula garante automaticamente que cada linha, cada coluna e cada
bloco 3x3 contenham os 9 digitos sem repeticao (propriedade verificavel
diretamente da aritmetica modular envolvida).

A partir desse tabuleiro-base fixo, as seguintes operacoes **sempre**
preservam a validade de um Sudoku (por isso podem ser aplicadas em qualquer
combinacao e ordem, com qualquer aleatoriedade):
1. Permutar os simbolos 1-9 (trocar todo "3" por "7", etc.)
2. Trocar duas linhas DENTRO da mesma faixa de 3 linhas
3. Trocar duas colunas DENTRO da mesma banda de 3 colunas
4. Trocar duas faixas de 3 linhas entre si
5. Trocar duas bandas de 3 colunas entre si
6. Transpor o tabuleiro (trocar linhas por colunas)

Aplicando essas operacoes em sequencia aleatoria, obtemos um tabuleiro final
que parece arbitrario mas e garantidamente valido — sem precisar de nenhuma
tentativa e erro.

## 1. Importações, constantes e utilitarios de impressao

In [1]:
import random
import numpy as np

TAMANHO = 9
TAMANHO_BLOCO = 3

DIFICULDADES = {
    "facil": 35,
    "medio": 45,
    "dificil": 55,
}


def linha(char="-", n=49):
    print(char * n)


def imprimir_tabuleiro(tabuleiro: np.ndarray, titulo_tab: str = ""):
    """Imprime o tabuleiro 9x9 separando os blocos 3x3 com espacamento extra.
    Celulas vazias (0) aparecem como '.'."""
    if titulo_tab:
        print(f"\n  {titulo_tab}")
    linha("-", 49)
    for i in range(TAMANHO):
        celulas = []
        for j in range(TAMANHO):
            valor = tabuleiro[i, j]
            celulas.append(str(valor) if valor != 0 else ".")
            if (j + 1) % TAMANHO_BLOCO == 0 and j != TAMANHO - 1:
                celulas.append("|")
        print("  " + " ".join(celulas))
        if (i + 1) % TAMANHO_BLOCO == 0 and i != TAMANHO - 1:
            linha("-", 49)
    linha("-", 49)

## 2. Classe `GeradorSudoku`

Gera tabuleiros 9x9 validos sem usar backtracking: parte de um
tabuleiro-base fixo (formula modular) e aplica uma sequencia de
transformacoes aleatorias que preservam a validade do Sudoku.

Essa escolha de design (em oposicao a busca com tentativa-e-erro) tem a
vantagem de ser **deterministica no tempo de execução**, não corre risco
de "travar" tentando preencher uma celula sem opções válidas.

In [2]:
class GeradorSudoku:
    def __init__(self, seed=None):
        self._rng = random.Random(seed)

    @staticmethod
    def _tabuleiro_base() -> np.ndarray:
        """base[i][j] = (deslocamento(i) + j) mod 9 + 1."""
        tabuleiro = np.zeros((TAMANHO, TAMANHO), dtype=int)
        for i in range(TAMANHO):
            deslocamento = TAMANHO_BLOCO * (i % TAMANHO_BLOCO) + (i // TAMANHO_BLOCO)
            for j in range(TAMANHO):
                tabuleiro[i, j] = (deslocamento + j) % TAMANHO + 1
        return tabuleiro

    # ---- Transformacoes que preservam validade ----

    def _permutar_simbolos(self, tabuleiro):
        simbolos = list(range(1, TAMANHO + 1))
        embaralhados = simbolos.copy()
        self._rng.shuffle(embaralhados)
        mapa = dict(zip(simbolos, embaralhados))
        return np.vectorize(mapa.get)(tabuleiro)

    def _trocar_linhas_na_faixa(self, tabuleiro):
        resultado = tabuleiro.copy()
        for faixa in range(TAMANHO_BLOCO):
            indices = list(range(faixa * 3, faixa * 3 + 3))
            embaralhados = indices.copy()
            self._rng.shuffle(embaralhados)
            resultado[indices, :] = tabuleiro[embaralhados, :]
        return resultado

    def _trocar_colunas_na_banda(self, tabuleiro):
        resultado = tabuleiro.copy()
        for banda in range(TAMANHO_BLOCO):
            indices = list(range(banda * 3, banda * 3 + 3))
            embaralhados = indices.copy()
            self._rng.shuffle(embaralhados)
            resultado[:, indices] = tabuleiro[:, embaralhados]
        return resultado

    def _trocar_faixas(self, tabuleiro):
        ordem_faixas = list(range(TAMANHO_BLOCO))
        self._rng.shuffle(ordem_faixas)
        blocos = [tabuleiro[f * 3:(f + 1) * 3, :] for f in ordem_faixas]
        return np.vstack(blocos)

    def _trocar_bandas(self, tabuleiro):
        ordem_bandas = list(range(TAMANHO_BLOCO))
        self._rng.shuffle(ordem_bandas)
        blocos = [tabuleiro[:, b * 3:(b + 1) * 3] for b in ordem_bandas]
        return np.hstack(blocos)

    def gerar_tabuleiro_completo(self) -> np.ndarray:
        """Aplica a sequencia completa de transformacoes sobre o tabuleiro-base."""
        tabuleiro = self._tabuleiro_base()
        tabuleiro = self._permutar_simbolos(tabuleiro)
        tabuleiro = self._trocar_linhas_na_faixa(tabuleiro)
        tabuleiro = self._trocar_colunas_na_banda(tabuleiro)
        tabuleiro = self._trocar_faixas(tabuleiro)
        tabuleiro = self._trocar_bandas(tabuleiro)
        if self._rng.random() < 0.5:
            tabuleiro = tabuleiro.T.copy()
        return tabuleiro

    # ---- Remocao de celulas (cria o "jogo" a partir da "resposta") ----

    def remover_celulas(self, tabuleiro_completo, quantidade) -> np.ndarray:
        """Remove 'quantidade' celulas aleatorias (define como 0), retornando uma copia."""
        jogo = tabuleiro_completo.copy()
        todas_posicoes = [(i, j) for i in range(TAMANHO) for j in range(TAMANHO)]
        self._rng.shuffle(todas_posicoes)
        for (i, j) in todas_posicoes[:quantidade]:
            jogo[i, j] = 0
        return jogo

    def gerar_jogo(self, dificuldade: str):
        """Gera o par (jogo_com_lacunas, resposta_completa) para a dificuldade dada."""
        if dificuldade not in DIFICULDADES:
            raise ValueError(
                f"Dificuldade '{dificuldade}' invalida. Use uma de: {list(DIFICULDADES)}."
            )
        resposta = self.gerar_tabuleiro_completo()
        jogo = self.remover_celulas(resposta, DIFICULDADES[dificuldade])
        return jogo, resposta

## 3. Validacao (usada nos testes para confirmar que a geracao esta correta)

In [3]:
def grupo_e_valido(grupo) -> bool:
    """Um grupo (linha, coluna ou bloco) e valido se contem 1..9 sem repeticao."""
    return sorted(grupo) == list(range(1, TAMANHO + 1))


def tabuleiro_e_valido(tabuleiro: np.ndarray) -> bool:
    """Verifica todas as 9 linhas, 9 colunas e 9 blocos 3x3 de uma vez."""
    for i in range(TAMANHO):
        if not grupo_e_valido(tabuleiro[i, :].tolist()):
            return False
    for j in range(TAMANHO):
        if not grupo_e_valido(tabuleiro[:, j].tolist()):
            return False
    for bi in range(TAMANHO_BLOCO):
        for bj in range(TAMANHO_BLOCO):
            bloco = tabuleiro[bi*3:bi*3+3, bj*3:bj*3+3].flatten().tolist()
            if not grupo_e_valido(bloco):
                return False
    return True


def contar_celulas_vazias(tabuleiro: np.ndarray) -> int:
    return int(np.sum(tabuleiro == 0))

## 4. Solver via backtracking (extra, complementar ao que foi pedido)

A questao pede apenas para **gerar** jogos e respostas, mas como
complemento mostramos que gerar e resolver sao operacoes logicamente
distintas: o gerador acima **nao** usa backtracking, mas o solver abaixo
usa, justamente para ilustrar a diferenca entre as duas abordagens dentro
do mesmo trabalho. Usa a heuristica **MRV (minimum remaining values)**:
em cada passo, escolhe a celula vazia com **menos candidatos possiveis**
(em vez de percorrer em ordem fixa) - isso poda bastante o espaco de busca
em relacao a um backtracking ingenuo posicao-por-posicao.

In [4]:
def _candidatos(tabuleiro, i, j) -> set:
    usados = set(tabuleiro[i, :]) | set(tabuleiro[:, j])
    bi, bj = (i // 3) * 3, (j // 3) * 3
    usados |= set(tabuleiro[bi:bi+3, bj:bj+3].flatten())
    return set(range(1, 10)) - usados


def resolver_com_backtracking(tabuleiro):
    """Resolve um Sudoku com lacunas via backtracking guiado por MRV.
    Retorna o tabuleiro resolvido, ou None se nao houver solucao."""
    tabuleiro = tabuleiro.copy()

    def passo() -> bool:
        melhor_pos = None
        melhor_candidatos = None
        for i in range(TAMANHO):
            for j in range(TAMANHO):
                if tabuleiro[i, j] == 0:
                    cands = _candidatos(tabuleiro, i, j)
                    if melhor_candidatos is None or len(cands) < len(melhor_candidatos):
                        melhor_pos, melhor_candidatos = (i, j), cands
                        if len(cands) == 0:
                            return False  # poda imediata: celula sem opcoes
        if melhor_pos is None:
            return True  # nenhuma celula vazia restante -> resolvido

        i, j = melhor_pos
        for valor in sorted(melhor_candidatos):
            tabuleiro[i, j] = valor
            if passo():
                return True
            tabuleiro[i, j] = 0
        return False

    return tabuleiro if passo() else None

## 5. Relatorio de um jogo

In [5]:
def relatorio_jogo(numero, jogo, resposta, dificuldade):
    vazias = contar_celulas_vazias(jogo)
    total = TAMANHO * TAMANHO
    print(f"\n{'=' * 49}")
    print(f"  JOGO {numero}  -  Dificuldade: {dificuldade.upper()}")
    print(f"{'=' * 49}")
    imprimir_tabuleiro(jogo, "Tabuleiro a resolver ('.' = celula vazia):")
    print(f"  -> {vazias} celulas vazias de {total} ({100 * vazias / total:.1f}% do tabuleiro)")
    imprimir_tabuleiro(resposta, "Resposta (tabuleiro completo):")
    print(f"  -> Resposta valida: {tabuleiro_e_valido(resposta)}")

## 6. Testes automatizados

Validamos: (1) o tabuleiro-base e valido por si so; (2) a cadeia completa
de transformacoes preserva validade; (3) tabuleiros sucessivos sao
diferentes; (4) cada dificuldade remove a quantidade certa de celulas;
(5) celulas nao removidas mantem o valor original; (6) dificuldade invalida
e rejeitada; (7) o solver extra resolve corretamente os jogos gerados.

In [6]:
def testes():
    print("\n" + "#" * 60)
    print("# BLOCO DE TESTES AUTOMATIZADOS -- QUESTAO 2 (SUDOKU)")
    print("#" * 60)

    gerador = GeradorSudoku(seed=7)

    print("\n--- Teste 1: tabuleiro-base (sem transformacoes) e valido ---")
    base = gerador._tabuleiro_base()
    assert tabuleiro_e_valido(base), "FALHOU: tabuleiro-base deveria ser valido"
    print("  OK: tabuleiro-base satisfaz linhas, colunas e blocos 3x3.")

    print("\n--- Teste 2: tabuleiro apos transformacoes completas e valido ---")
    for tentativa in range(20):
        completo = gerador.gerar_tabuleiro_completo()
        assert tabuleiro_e_valido(completo), (
            f"FALHOU na tentativa {tentativa}: tabuleiro invalido apos transformacoes"
        )
    print("  OK: 20 tabuleiros gerados em sequencia, todos validos.")

    print("\n--- Teste 3: tabuleiros gerados sao diferentes entre execucoes ---")
    t1 = gerador.gerar_tabuleiro_completo()
    t2 = gerador.gerar_tabuleiro_completo()
    assert not np.array_equal(t1, t2), "FALHOU: dois tabuleiros gerados sao identicos"
    print("  OK: tabuleiros consecutivos sao diferentes (aleatoriedade funcionando).")

    print("\n--- Teste 4: quantidade de celulas removidas por dificuldade ---")
    for dif, esperado in DIFICULDADES.items():
        jogo, resposta = gerador.gerar_jogo(dif)
        vazias = contar_celulas_vazias(jogo)
        assert vazias == esperado, f"FALHOU: {dif} deveria remover {esperado}, removeu {vazias}"
        assert tabuleiro_e_valido(resposta), f"FALHOU: resposta de '{dif}' nao e valida"
        print(f"  {dif:8s}: {vazias} celulas removidas (esperado {esperado}) -> OK")

    print("\n--- Teste 5: celulas nao removidas mantem o valor original ---")
    jogo5, resposta5 = gerador.gerar_jogo("medio")
    preservadas = (jogo5 != 0)
    assert np.array_equal(jogo5[preservadas], resposta5[preservadas]), (
        "FALHOU: alguma celula preservada foi alterada na remocao"
    )
    print("  OK: todas as celulas nao removidas continuam iguais a resposta.")

    print("\n--- Teste 6: dificuldade invalida e rejeitada ---")
    try:
        gerador.gerar_jogo("impossivel")
        raise AssertionError("FALHOU: deveria ter lancado ValueError")
    except ValueError as e:
        print(f"  OK: erro tratado corretamente -> {e}")

    print("\n--- Teste 7: solver por backtracking resolve os jogos gerados ---")
    for dif in DIFICULDADES:
        jogo7, resposta7 = gerador.gerar_jogo(dif)
        resolvido = resolver_com_backtracking(jogo7)
        assert resolvido is not None, f"FALHOU: solver nao encontrou solucao para '{dif}'"
        assert tabuleiro_e_valido(resolvido), f"FALHOU: solucao do solver invalida para '{dif}'"
        print(f"  {dif:8s}: solver encontrou uma solucao valida -> OK")

    print("\n" + "#" * 60)
    print("# TODOS OS TESTES PASSARAM")
    print("#" * 60 + "\n")


testes()


############################################################
# BLOCO DE TESTES AUTOMATIZADOS -- QUESTAO 2 (SUDOKU)
############################################################

--- Teste 1: tabuleiro-base (sem transformacoes) e valido ---
  OK: tabuleiro-base satisfaz linhas, colunas e blocos 3x3.

--- Teste 2: tabuleiro apos transformacoes completas e valido ---
  OK: 20 tabuleiros gerados em sequencia, todos validos.

--- Teste 3: tabuleiros gerados sao diferentes entre execucoes ---
  OK: tabuleiros consecutivos sao diferentes (aleatoriedade funcionando).

--- Teste 4: quantidade de celulas removidas por dificuldade ---
  facil   : 35 celulas removidas (esperado 35) -> OK
  medio   : 45 celulas removidas (esperado 45) -> OK
  dificil : 55 celulas removidas (esperado 55) -> OK

--- Teste 5: celulas nao removidas mantem o valor original ---
  OK: todas as celulas nao removidas continuam iguais a resposta.

--- Teste 6: dificuldade invalida e rejeitada ---
  OK: erro tratado corretame

## 7. Demonstracao: gerar um jogo de cada dificuldade

In [7]:
def demonstracao():
    """Gera um jogo de cada dificuldade e imprime o relatorio completo."""
    print("\n" + "=" * 60)
    print("  QUESTAO 2 -- GERADOR DE SUDOKU  |  DEMONSTRACAO")
    print("=" * 60)
    gerador = GeradorSudoku()
    for numero, dificuldade in enumerate(DIFICULDADES, start=1):
        jogo, resposta = gerador.gerar_jogo(dificuldade)
        relatorio_jogo(numero, jogo, resposta, dificuldade)
    print(f"\n{'=' * 60}")
    print("  FIM DA DEMONSTRACAO")
    print(f"{'=' * 60}\n")


demonstracao()


  QUESTAO 2 -- GERADOR DE SUDOKU  |  DEMONSTRACAO

  JOGO 1  -  Dificuldade: FACIL

  Tabuleiro a resolver ('.' = celula vazia):
-------------------------------------------------
  2 . . | 3 . . | 7 . 5
  . 7 5 | 2 9 6 | 8 1 .
  3 1 . | 5 . 7 | 9 2 .
-------------------------------------------------
  . 3 . | 7 . . | . . .
  5 4 7 | 6 2 9 | 1 . .
  6 2 9 | . . 1 | 4 5 .
-------------------------------------------------
  1 8 3 | . . . | 6 9 .
  7 . . | 9 6 . | 3 8 .
  . 6 . | 1 . 3 | . . 4
-------------------------------------------------
  -> 35 celulas vazias de 81 (43.2% do tabuleiro)

  Resposta (tabuleiro completo):
-------------------------------------------------
  2 9 6 | 3 1 8 | 7 4 5
  4 7 5 | 2 9 6 | 8 1 3
  3 1 8 | 5 4 7 | 9 2 6
-------------------------------------------------
  8 3 1 | 7 5 4 | 2 6 9
  5 4 7 | 6 2 9 | 1 3 8
  6 2 9 | 8 3 1 | 4 5 7
-------------------------------------------------
  1 8 3 | 4 7 5 | 6 9 2
  7 5 4 | 9 6 2 | 3 8 1
  9 6 2 | 1 8 3 | 5 7 4
----

## 8. Modo interativo (pede input do usuario)

Descomente e rode a celula abaixo para gerar jogos sob demanda, escolhendo
quantidade e dificuldade.

In [8]:
def main():
    """Modo interativo: pergunta quantidade de jogos e dificuldade desejadas."""
    print("\n" + "=" * 60)
    print("  QUESTAO 2 -- GERADOR DE SUDOKU")
    print("=" * 60)

    while True:
        try:
            quantidade = int(input("\nQuantos jogos deseja gerar? "))
            if quantidade < 1:
                print("  Informe um numero maior que zero.")
                continue
            break
        except ValueError:
            print("  Entrada invalida. Informe um numero inteiro.")

    while True:
        dificuldade = input("Dificuldade? [facil / medio / dificil]: ").strip().lower()
        if dificuldade in DIFICULDADES:
            break
        print("  Opcao invalida. Digite: facil, medio ou dificil.")

    gerador = GeradorSudoku()
    for numero in range(1, quantidade + 1):
        jogo, resposta = gerador.gerar_jogo(dificuldade)
        relatorio_jogo(numero, jogo, resposta, dificuldade)


# main()  # descomente esta linha para rodar o modo interativo